# Simple U-Net Denoiser (STL-10, native 96x96) + Cross-Noise-Level Sweep

Trains `SmallUNet` (`code/stl10_unet.py`) as a denoiser on STL-10's
native **96x96x3** images at ONE fixed noise_std (the **source std**,
`--noise-std`, training itself is unchanged from the plain version).
After training, the model is **tested at an array of different, fixed
noise_std values spanning `[0, 1]`**, and every sweep point's result is
reported **relative to the model's own result at its source std**:

```
mse_ratio_vs_source(sigma)   = test_mse(sigma)  / test_mse(source_std)   (= 1.0 at sigma = source_std)
delta_psnr_vs_source(sigma)  = test_psnr(sigma) - test_psnr(source_std)  (= 0.0 dB at sigma = source_std)
```

This directly answers: *how much worse (or better) does the network
do on noise levels it wasn't trained for, compared to the noise level
it WAS trained for?*

Core code lives in `code/unet_denoising_simple_stl10.py`
(version-controlled); this notebook is a thin orchestration layer: GPU
check, repo pull, build model + print the full shape trace, download
real STL-10 (cached in Drive), train at the source std, plot the
training loss curve, then run and plot the cross-noise-level sweep
(quantitative table + curve, plus a 3-image qualitative grid) as the
required results-visualization step.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model code lives in `code/stl10_unet.py` (architecture, reused as-is)
and `code/unet_denoising_simple_stl10.py` (denoising train/eval loop,
cross-noise-level sweep, visualization), not duplicated in this
notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model, print full shape trace

One verbose forward pass on a small synthetic 96x96 batch, printing the
`[B,C,H,W]` shape after every op -- including, per level, each skip
tensor's shape at the moment it is saved and the shapes immediately
before/after each `torch.cat`. Not part of training.

In [ ]:
from stl10_unet import SmallUNet, count_trainable_parameters

torch.manual_seed(0)

model = SmallUNet().to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.rand(4, 3, 96, 96, device=device)  # [B,3,96,96], synthetic, in [0,1]
_ = model(sample_batch, verbose=True)


## 4. Download real STL-10 (train/test split), native 96x96, cached in Drive

`get_stl10_dataloaders` (from `code/stl10_unet.py`) uses `torchvision`'s
native labeled train/test partition at STL-10's native 96x96
resolution -- no resizing. Labels are unused (the denoising target is
the clean image itself). STL-10 is a ~2.6GB download, so Drive caching
matters even more here than for CIFAR-10.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so we mount Google Drive and point `DATA_ROOT` at a
persistent Drive folder: `torchvision` downloads into it once and
reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/stl10"


## 5. Train at the source noise_std

Explicit train/eval loops (`train_one_epoch`, `evaluate`, both defined
in `code/unet_denoising_simple_stl10.py`): each clean batch is
corrupted at the SAME fixed `SOURCE_STD` throughout training (no
randomization -- training itself is unchanged from the plain version).
MSE loss, Adam optimizer.

In [ ]:
import torch.nn as nn
from unet_denoising_simple_stl10 import train_one_epoch, evaluate
from stl10_unet import get_stl10_dataloaders

BATCH_SIZE = 64
N_EPOCHS = 15
LR = 1e-3
SOURCE_STD = 0.3  # the single noise_std the model is trained at

train_loader, test_loader = get_stl10_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "test_denoising_mse": []}

for epoch in range(N_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device, SOURCE_STD)
    test_mse = evaluate(model, test_loader, loss_fn, device, SOURCE_STD)

    history["train_loss"].append(train_loss)
    history["test_denoising_mse"].append(test_mse)

    print(f"epoch {epoch + 1:3d}/{N_EPOCHS}  train_loss={train_loss:.4f}  test_denoising_mse={test_mse:.4f}")


## 6. Loss curve (quantitative result over training, at the source std)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train_loss")
ax.plot(history["test_denoising_mse"], label="test_denoising_mse")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE(x_hat, clean)")
ax.set_title(f"Denoising loss (source_std={SOURCE_STD})")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_denoising_mse @ source_std={SOURCE_STD}: {history['test_denoising_mse'][-1]:.4f}")


## 7. Qualitative check at the source std

5 real test images as clean (row 1) / noisy input (row 2) / U-Net
output (row 3), with per-image MSE/PSNR (output vs. clean) -- this is
the plain, single-noise-level result (same as the non-sweep version of
this script).

In [ ]:
from unet_denoising_simple_stl10 import visualize_denoising

sample_images, _sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_denoising(model, sample_images, SOURCE_STD, num_examples=5)


## 8. Cross-noise-level sweep: testing at DIFFERENT std values, relative to the source

The model was trained at exactly one std (`SOURCE_STD`). Here it is
evaluated on the whole test set at several OTHER fixed std values
spanning `[0, 1]`, and every result is expressed relative to the
model's own result at `SOURCE_STD`:

- `mse_ratio_vs_source` = 1.0 at `sigma = SOURCE_STD` by construction;
  values above 1.0 mean denoising is worse at that sigma than at the
  std the model was actually trained for.
- `delta_psnr_vs_source` = 0.0 dB at `sigma = SOURCE_STD`; negative
  values mean worse PSNR than at the source std.

`STD_SWEEP_VALUES` always includes `SOURCE_STD` itself (added
automatically if missing), since it is required as the baseline every
other point is measured against.

In [ ]:
from unet_denoising_simple_stl10 import (
    evaluate_std_sweep,
    summarize_std_sweep_relative_to_source,
    print_std_sweep_summary,
    plot_std_sweep,
)

STD_SWEEP_VALUES = sorted({0.0, 0.2, 0.4, 0.6, 0.8, 1.0} | {SOURCE_STD})

sweep_mse = evaluate_std_sweep(model, test_loader, loss_fn, device, STD_SWEEP_VALUES)
summary = summarize_std_sweep_relative_to_source(sweep_mse, SOURCE_STD)
print_std_sweep_summary(summary, SOURCE_STD)

plot_std_sweep(summary, SOURCE_STD)


## 9. Qualitative sweep: clean vs. denoised output at each fixed noise_std, vs. source

3 real test images, one row per sigma in `STD_SWEEP_VALUES` (plus a
clean reference row). The `SOURCE_STD` row is labeled `[source]`; every
other row's title includes `vs. source: <delta_mse>` -- that same
image's MSE at this sigma minus its MSE at `SOURCE_STD` -- so
degradation (or lack of it) away from the training std can be read off
directly, per image.

In [ ]:
from unet_denoising_simple_stl10 import visualize_std_sweep_examples

visualize_std_sweep_examples(model, sample_images, STD_SWEEP_VALUES, SOURCE_STD, num_examples=3)
